# 04 · Compose insight summary

Reads the 6 result tables + cluster profile and emits `insight_summary.md` — the 1-page judge-facing deliverable.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from pathlib import Path
from config.settings import DATA_RESULTS, PROJECT_ROOT

flavor = pd.read_csv(DATA_RESULTS / 'flavor_ranking.csv')
pain = pd.read_csv(DATA_RESULTS / 'pain_ranking.csv')
occ = pd.read_csv(DATA_RESULTS / 'occasion_distribution.csv')
fmt = pd.read_csv(DATA_RESULTS / 'format_appeal.csv')
comp = pd.read_csv(DATA_RESULTS / 'competitor_scorecard.csv')
clusters = pd.read_csv(DATA_RESULTS / 'cluster_profiles.csv')

total = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'twitter_enriched.csv').shape[0]
print(f'Total clean tweets: {total}')

Total clean tweets: 5021


In [2]:
lines = []
lines.append('# KSF Consumer Intelligence — Insight Summary')
lines.append('')
lines.append(f'**Total clean tweets analyzed:** {total}  ')
lines.append('**Source:** Twitter/X (English) · Nov 2025 – May 2026  ')
lines.append('**Pipeline:** Playwright network interception → VADER+TextBlob sentiment → entity extraction → K-means segmentation  ')
lines.append('')
lines.append('---')
lines.append('')

# Q1
top5_pos = flavor.sort_values('net_score', ascending=False).head(5)
top5_neg = flavor.sort_values('net_score', ascending=True).head(3)
lines.append('## Q1 · Flavor preferences')
lines.append('**Top 5 most-desired:**')
for _, r in top5_pos.iterrows():
    lines.append(f"- **{r['flavor'].replace('_',' ').title()}** — {int(r['mentions'])} mentions, net {r['net_score']:+.0f}%")
lines.append('')
lines.append('**Most-rejected:**')
for _, r in top5_neg.iterrows():
    lines.append(f"- {r['flavor'].replace('_',' ').title()} — net {r['net_score']:+.0f}%")
lines.append('')
lines.append('![Q1](outputs/figures/Q1_flavor_diverging.png)')
lines.append('')

# Q2 — morning summary numbers come from the occasion table
morning_row = occ[occ['occasion'] == 'morning']
morning_pct = float(morning_row['pct_of_total'].iloc[0]) if not morning_row.empty else 0
lines.append('## Q2 · Morning routine')
lines.append(f'Morning/breakfast accounts for **{morning_pct:.0f}%** of occasion-tagged tweets — strongest single occasion. See `outputs/figures/Q2_morning_apac.png` for APAC vs Global split and monthly trend.')
lines.append('')

# Q3
top3_pain = pain.head(3)
lines.append('## Q3 · Sensory pain points')
for _, r in top3_pain.iterrows():
    lines.append(f"- **{r['pain'].replace('_',' ').title()}** — {int(r['mentions'])} mentions, severity {r['severity']:.2f}")
lines.append('')

# Q4
yog_row = fmt[fmt['format'] == 'yogurt_drink']
if not yog_row.empty:
    yr = yog_row.iloc[0]
    lines.append('## Q4 · Yogurt-drink appetite')
    lines.append(f"Yogurt drink: **{int(yr['mentions'])} mentions**, net sentiment **{yr['net_score']:+.0f}%** ({yr['pct_of_mentions']:.0f}% of format mentions).")
    lines.append('')

# Q5
lines.append('## Q5 · Occasion distribution')
for _, r in occ.head(5).iterrows():
    lines.append(f"- {r['occasion'].replace('_',' ').title()}: {int(r['mentions'])} ({r['pct_of_total']:.0f}%)")
lines.append('')

# Q6
lines.append('## Q6 · Competitor scorecards')
for _, r in comp.head(5).iterrows():
    lines.append(f"- **{r['brand']}**: {int(r['mentions'])} mentions, net {r['net_score']:+.0f}%")
lines.append('')

# Clusters
lines.append('## BONUS · Consumer segments (k-means)')
for _, r in clusters.iterrows():
    lines.append(f"- **C{int(r['cluster'])} — {r['label']}** ({r['size_pct']:.0f}% of corpus) · flavor: {r['top_flavor']}, pain: {r['top_pain']}, occasion: {r['top_occasion']}")
lines.append('')
lines.append('---')
lines.append('')
lines.append('## Strategy implication')
lines.append('Direct cause→effect statement to integrate into pitch:')
lines.append(f'> Consumers ranked **{top5_pos.iloc[0]["flavor"]}** as the highest net-sentiment flavor and flagged **{top3_pain.iloc[0]["pain"]}** as the dominant pain. We designed our protein yogurt RTD to lead with this flavor and eliminate this pain.')

out = PROJECT_ROOT / 'insight_summary.md'
out.write_text('\n'.join(lines), encoding='utf-8')
print(f'Wrote {out}')
print('\n--- Preview ---\n')
print('\n'.join(lines[:30]))

Wrote /home/azril/Personal/Projects/DSAI/NUSFTC/nlp social listening 2/insight_summary.md

--- Preview ---

# KSF Consumer Intelligence — Insight Summary

**Total clean tweets analyzed:** 5021  
**Source:** Twitter/X (English) · Nov 2025 – May 2026  
**Pipeline:** Playwright network interception → VADER+TextBlob sentiment → entity extraction → K-means segmentation  

---

## Q1 · Flavor preferences
**Top 5 most-desired:**
- **Taro** — 5 mentions, net +100%
- **Passion Fruit** — 1 mentions, net +100%
- **Jasmine** — 2 mentions, net +100%
- **Black Sesame** — 3 mentions, net +67%
- **Mango** — 73 mentions, net +53%

**Most-rejected:**
- Yuzu — net -100%
- Lychee — net +0%
- Strawberry — net +6%

![Q1](outputs/figures/Q1_flavor_diverging.png)

## Q2 · Morning routine
Morning/breakfast accounts for **67%** of occasion-tagged tweets — strongest single occasion. See `outputs/figures/Q2_morning_apac.png` for APAC vs Global split and monthly trend.

## Q3 · Sensory pain points
- **Chalky** — 2